# 🏢 Phase 4 — Extraction d'Entités Nommées (NER)

**Projet : African Media Intelligence AI**  
**Auteur : Esmel Amari (Phil)**  
**Description :** Ce notebook extrait automatiquement les entités nommées des articles médiatiques : entreprises, organisations, pays, villes et personnes publiques. Ces entités serviront de base au scoring de réputation en Phase 5.

---

### Entités ciblées
| Type | Label spaCy | Exemples |
|---|---|---|
| Organisation | `ORG` | CEDEAO, Orange CI, MTN, Banque Mondiale |
| Lieu géopolitique | `GPE` | Côte d'Ivoire, Nigéria, Accra |
| Personne | `PER` | Alassane Ouattara, Macky Sall |
| Localisation | `LOC` | Sahel, Golfe de Guinée |


## 0. Imports & Configuration

In [1]:
import pandas as pd
import numpy as np
import spacy
import subprocess
import json
import re
import logging
import warnings
from pathlib import Path
from datetime import datetime
from collections import Counter, defaultdict

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger(__name__)

# ── Chemins ────────────────────────────────────────────────────────
BASE_DIR = Path('..').resolve()
PROC_DIR = BASE_DIR / 'data' / 'processed'
PROC_DIR.mkdir(parents=True, exist_ok=True)

print("✅ Imports chargés")

✅ Imports chargés


## 1. Installation & Chargement du Modèle spaCy

In [2]:
# ── Téléchargement du modèle spaCy français ───────────────────────
# Modèle fr_core_news_md = taille moyenne, meilleur NER pour le français
try:
    nlp_fr = spacy.load('fr_core_news_md')
    print("✅ Modèle 'fr_core_news_md' chargé")
except OSError:
    print("⏳ Téléchargement du modèle français...")
    subprocess.run(['python', '-m', 'spacy', 'download', 'fr_core_news_md'], check=True)
    nlp_fr = spacy.load('fr_core_news_md')
    print("✅ Modèle 'fr_core_news_md' installé et chargé")

# ── Modèle anglais ────────────────────────────────────────────────
try:
    nlp_en = spacy.load('en_core_web_sm')
    print("✅ Modèle 'en_core_web_sm' chargé")
except OSError:
    print("⏳ Téléchargement du modèle anglais...")
    subprocess.run(['python', '-m', 'spacy', 'download', 'en_core_web_sm'], check=True)
    nlp_en = spacy.load('en_core_web_sm')
    print("✅ Modèle 'en_core_web_sm' installé et chargé")

print(f"\n📋 Labels NER FR : {nlp_fr.get_pipe('ner').labels}")
print(f"📋 Labels NER EN : {nlp_en.get_pipe('ner').labels}")

✅ Modèle 'fr_core_news_md' chargé
✅ Modèle 'en_core_web_sm' chargé

📋 Labels NER FR : ('LOC', 'MISC', 'ORG', 'PER')
📋 Labels NER EN : ('CARDINAL', 'DATE', 'EVENT', 'FAC', 'GPE', 'LANGUAGE', 'LAW', 'LOC', 'MONEY', 'NORP', 'ORDINAL', 'ORG', 'PERCENT', 'PERSON', 'PRODUCT', 'QUANTITY', 'TIME', 'WORK_OF_ART')


## 2. Chargement des Données

In [3]:
# Priorité : fichier avec topics > fichier avec sentiment > fichier raw
for pattern in ['articles_topics_*.csv', 'articles_sentiment_*.csv', 'articles_raw_*.csv']:
    search_dir = PROC_DIR if 'raw' not in pattern else BASE_DIR / 'data' / 'raw'
    files = sorted(PROC_DIR.glob(pattern), reverse=True)
    if not files:
        files = sorted((BASE_DIR / 'data' / 'raw').glob(pattern), reverse=True)
    if files:
        df = pd.read_csv(files[0], encoding='utf-8-sig')
        print(f"✅ Chargé : {files[0].name} → {df.shape[0]} articles")
        break
else:
    raise FileNotFoundError("Aucun fichier trouvé. Exécutez les notebooks précédents.")

✅ Chargé : articles_topics_20260512_2031.csv → 121 articles


## 3. Dictionnaire d'Entités Africaines (Boost)

In [4]:
# ── Dictionnaire manuel pour améliorer la détection spaCy ─────────
# Ces entités africaines sont souvent mal reconnues par les modèles génériques

ENTITIES_BOOST = {
    'ORG': [
        # Institutions régionales
        'CEDEAO', 'ECOWAS', 'UA', 'Union Africaine', 'African Union',
        'BCEAO', 'UEMOA', 'OHADA', 'BAD', 'CFA',
        # Banques & Finance
        'Banque Mondiale', 'World Bank', 'FMI', 'IMF',
        'Ecobank', 'UBA', 'Coris Bank', 'BGFI', 'SIB', 'BICICI',
        # Télécoms & Tech
        'MTN', 'Orange CI', 'Moov Africa', 'Airtel', 'Safaricom',
        'Jumia', 'Flutterwave', 'Wave', 'M-Pesa', 'Paystack',
        # Énergie & Ressources
        'Total Energies', 'Tullow Oil', 'PETROCI',
        # Médias
        'RFI', 'BBC Afrique', 'Jeune Afrique', 'AllAfrica',
    ],
    'GPE': [
        # Pays
        "Côte d'Ivoire", 'Sénégal', 'Ghana', 'Nigéria', 'Nigeria',
        'Mali', 'Burkina Faso', 'Guinée', 'Cameroun', 'Kenya',
        'Éthiopie', 'Afrique du Sud', 'Maroc', 'Égypte', 'Tanzanie',
        'Rwanda', 'Ouganda', 'Mozambique', 'Zambie', 'Zimbabwe',
        'Togo', 'Bénin', 'Niger', 'Tchad', 'Soudan', 'Libye',
        # Capitales & villes
        'Abidjan', 'Dakar', 'Accra', 'Lagos', 'Nairobi', 'Addis-Abeba',
        'Johannesburg', 'Casablanca', 'Le Caire', 'Bamako', 'Ouagadougou',
    ],
    'PER': [
        'Alassane Ouattara', 'Macky Sall', 'Nana Akufo-Addo',
        'Bola Tinubu', 'William Ruto', 'Paul Kagame',
        'Cyril Ramaphosa', 'Mohammed VI', 'Abdel Fattah el-Sissi',
    ]
}

# Construction d'un set pour recherche rapide
ALL_BOOST_ENTITIES = {}
for etype, entities in ENTITIES_BOOST.items():
    for ent in entities:
        ALL_BOOST_ENTITIES[ent.lower()] = (ent, etype)

print(f"✅ Dictionnaire boost : {len(ALL_BOOST_ENTITIES)} entités africaines")

✅ Dictionnaire boost : 83 entités africaines


## 4. Fonction d'Extraction NER

In [5]:
# Labels cibles
TARGET_LABELS = {'ORG', 'GPE', 'PER', 'LOC', 'PERSON', 'FAC'}

# Mapping labels EN → FR standard
LABEL_NORMALIZE = {
    'PERSON' : 'PER',
    'FAC'    : 'ORG',
    'NORP'   : 'GPE',
}

def extract_entities(text: str, langue: str = 'fr') -> list:
    """
    Extrait les entités nommées d'un texte.

    Retourne : liste de dicts {entite, type, source}
    """
    if not isinstance(text, str) or len(text.strip()) < 10:
        return []

    entities = []
    text_clean = text[:2000]  # Limite de traitement

    # ── Étape 1 : Dictionnaire boost (entités africaines) ──────────
    text_lower = text_clean.lower()
    for key, (nom, etype) in ALL_BOOST_ENTITIES.items():
        if key in text_lower:
            entities.append({'entite': nom, 'type': etype, 'source': 'boost'})

    # ── Étape 2 : spaCy NER ────────────────────────────────────────
    nlp = nlp_fr if langue == 'fr' else nlp_en
    doc = nlp(text_clean)

    for ent in doc.ents:
        label = LABEL_NORMALIZE.get(ent.label_, ent.label_)
        if label not in TARGET_LABELS:
            continue
        nom = ent.text.strip()
        # Filtrer les entités trop courtes ou numériques
        if len(nom) < 3 or nom.isdigit():
            continue
        # Filtrer les stop words courants mal classifiés
        if nom.lower() in {'lundi','mardi','mercredi','jeudi','vendredi',
                            'ce','cette','les','des','une','monday','tuesday'}:
            continue
        entities.append({'entite': nom, 'type': label, 'source': 'spacy'})

    # Dédupliquer (nom + type)
    seen = set()
    unique_entities = []
    for e in entities:
        key = (e['entite'].lower(), e['type'])
        if key not in seen:
            seen.add(key)
            unique_entities.append(e)

    return unique_entities


# Test
test = "La CEDEAO a convoqué un sommet d'urgence à Abidjan suite aux tensions au Mali. Orange CI et MTN ont annoncé un partenariat."
result = extract_entities(test, 'fr')
print("🔬 Test NER :")
for e in result:
    print(f"  [{e['type']}] {e['entite']} ({e['source']})")

🔬 Test NER :
  [ORG] CEDEAO (boost)
  [ORG] MTN (boost)
  [ORG] Orange CI (boost)
  [GPE] Mali (boost)
  [GPE] Abidjan (boost)
  [LOC] Abidjan (spacy)
  [LOC] Mali (spacy)


## 5. Application sur le Dataset

In [6]:
print(f"🔄 Extraction NER sur {len(df)} articles...")

# Appliquer article par article
all_article_entities = []
entities_flat = []  # Pour table long-format (Power BI)

for idx, row in df.iterrows():
    texte = str(row.get('titre', '')) + ' ' + str(row.get('resume', ''))
    langue = str(row.get('langue', 'fr'))[:2]
    entities = extract_entities(texte, langue)

    # Format JSON pour colonne dans df principal
    all_article_entities.append(json.dumps(entities, ensure_ascii=False))

    # Format long pour analyse
    for e in entities:
        entities_flat.append({
            'article_id'       : row.get('id', idx),
            'source'           : row.get('source', ''),
            'date_publication' : row.get('date_publication', ''),
            'langue'           : langue,
            'sentiment'        : row.get('sentiment', ''),
            'topic_label'      : row.get('topic_label', ''),
            'entite'           : e['entite'],
            'type_entite'      : e['type'],
            'detection_source' : e['source']
        })

    if (idx + 1) % 50 == 0:
        print(f"\r  Progression : {idx+1}/{len(df)}", end='')

print(f"\n\n✅ NER terminé")
print(f"   → {len(entities_flat)} mentions d'entités au total")

# Ajouter colonne JSON dans df principal
df['entites_json'] = all_article_entities

🔄 Extraction NER sur 121 articles...
  Progression : 100/121

✅ NER terminé
   → 532 mentions d'entités au total


## 6. Analyse des Entités Extraites

In [7]:
df_entities = pd.DataFrame(entities_flat)

if df_entities.empty:
    print("⚠️  Aucune entité extraite. Vérifiez le texte des articles.")
else:
    print("=== DISTRIBUTION PAR TYPE D'ENTITÉ ===")
    print(df_entities['type_entite'].value_counts().to_string())

    print("\n=== TOP 20 ORGANISATIONS CITÉES ===")
    orgs = df_entities[df_entities['type_entite'] == 'ORG']
    print(orgs['entite'].value_counts().head(20).to_string())

    print("\n=== TOP 15 PAYS/VILLES CITÉS ===")
    gpe = df_entities[df_entities['type_entite'] == 'GPE']
    print(gpe['entite'].value_counts().head(15).to_string())

    print("\n=== TOP 10 PERSONNES CITÉES ===")
    pers = df_entities[df_entities['type_entite'] == 'PER']
    print(pers['entite'].value_counts().head(10).to_string())

=== DISTRIBUTION PAR TYPE D'ENTITÉ ===
type_entite
LOC    164
ORG    141
GPE    126
PER    101

=== TOP 20 ORGANISATIONS CITÉES ===
entite
UA                                                    21
Africa Forward                                         9
BBC                                                    5
Monde                                                  4
Capital FM] Nairobi                                    4
JNIM                                                   3
UN News                                                2
The National Assembly Committee on Trade, Industry     1
International Football                                 1
Jeunes diplômés                                        1
FLA mouvement séparatiste touareg                      1
Al-Qaïda                                               1
Service mondial de la                                  1
UBA                                                    1
IFAB                                                   1
minist

## 7. Sauvegarde

In [8]:
timestamp = datetime.now().strftime('%Y%m%d_%H%M')

# ── Articles enrichis ──────────────────────────────────────────────
main_file = PROC_DIR / f'articles_ner_{timestamp}.csv'
df.to_csv(main_file, index=False, encoding='utf-8-sig')
print(f"✅ Articles enrichis : {main_file}")

# ── Table entités (long format — base du RepScore) ─────────────────
if not df_entities.empty:
    entities_file = PROC_DIR / f'entities_flat_{timestamp}.csv'
    df_entities.to_csv(entities_file, index=False, encoding='utf-8-sig')
    print(f"✅ Entités long format : {entities_file}")
    print(f"   → {len(df_entities)} lignes | {df_entities['entite'].nunique()} entités uniques")

✅ Articles enrichis : C:\Users\E682\Desktop\data\processed\articles_ner_20260512_2037.csv
✅ Entités long format : C:\Users\E682\Desktop\data\processed\entities_flat_20260512_2037.csv
   → 532 lignes | 314 entités uniques


---

## ✅ Phase 4 Terminée

**Ce qui a été accompli :**
- ✅ Modèles spaCy FR + EN chargés
- ✅ Dictionnaire boost de 50+ entités africaines
- ✅ NER appliqué sur tous les articles (ORG, GPE, PER, LOC)
- ✅ Table long-format des entités exportée

**Prochaine étape :** `05_reputation_scoring.ipynb` — Calcul du score de réputation par entité
